---
icon: lucide/ruler
---

# Physical Units

Every array in a simulation holds physical quantities, but a JAX array only stores numbers. <em>k</em>UPS has so far relied on a convention: lengths in Å, energies in eV, masses in amu, charges in $e$, and temperatures in K. The convention is cheap, but it is not checked. A force field parameterized in kcal/mol, a pressure given in bar, or a forgotten factor of $k_B$ produces numbers that look plausible and are wrong.

[kups.core.units][kups.core.units] makes the convention explicit. A [Quantity][kups.core.units.Quantity] is an array together with its physical dimension. Dimensions are checked when JAX traces a function, so a mismatch raises before any computation runs. They also live in the static types, so [pyrefly](https://pyrefly.org) finds many mistakes before the code is executed. Because the dimension is pytree metadata and not data, a compiled function is identical to one written with plain arrays.

In [ ]:
import jax
import jax.numpy as jnp

from kups.core import units
from kups.core.units import catalog as u
from kups.core.units import constants as const

jax.config.update("jax_enable_x64", True)

## Quantities

Multiplying a number or an array by a unit creates a quantity. Arithmetic propagates the dimension, `.to(unit)` returns the plain magnitudes in any compatible unit, and adding incompatible quantities fails with a [DimensionError][kups.core.units.DimensionError].

In [ ]:
r = jnp.array([3.4, 3.8, 4.5]) * u.angstrom
print(r)
print(r.to(u.nm))

energy = 0.5 * u.kcal / u.mol + 0.01 * u.eV
print(energy)

In [ ]:
try:
    r + 1.0 * u.eV
except units.DimensionError as err:
    print(err)

## Base Systems

A quantity stores its magnitudes in a *base system*. The default, `ATOMISTIC`, uses exactly the <em>k</em>UPS convention: Å, amu, eV, $e$, and K. Therefore `q.value` is the number existing <em>k</em>UPS code already expects, and moving between a quantity and a plain array never introduces a conversion factor. Derived dimensions are expressed in the same base. Time, for instance, is measured in $\text{Å}\sqrt{\text{amu}/\text{eV}} \approx 10.18$ fs.

In [ ]:
dt = 0.5 * u.fs
print(dt.value)  # magnitude in the base time unit
print(dt.to(u.fs))

with units.base(units.SI):
    print(3.0 * u.angstrom)

Keeping magnitudes in the atomistic base matters for precision. In SI, an energy of 1 eV is $1.6 \times 10^{-19}$ J, close to where single precision starts losing accuracy. In the atomistic base, typical simulation values are of order one in float32 and float64 alike.

## Units from Text

Configuration files store quantities as strings. [parse][kups.core.units.catalog.parse] reads a number followed by a unit expression. Every unit accepts the SI prefixes, and anything a quantity prints can be parsed back.

In [ ]:
print(u.parse("1 bar").to(u.eV / u.angstrom**3))
print(u.parse("1 kcal/mol/angstrom").to(u.eV / u.angstrom))
print(u.parse("120 kJ/mol").to(u.eV))

[constants][kups.core.units.constants] provides CODATA 2018 values as quantities, so factors such as $k_B$ carry their dimension.

In [ ]:
kT = const.k_B * (300.0 * u.K)
print(kT)
print(const.k_e)  # Coulomb constant

## Gradients

Forces are negative gradients of the energy with respect to positions, so their dimension is energy per length. `jax.grad` cannot know this: it gives a cotangent the dimension of the input. [units.grad][kups.core.units.grad] assigns the derivative the dimension $\dim(\text{out}) / \dim(\text{arg})$. The same holds for `value_and_grad`, `vjp`, `jacrev`, `jacfwd`, and `hessian`.

In [ ]:
def lennard_jones(r, epsilon, sigma):
    s6 = (sigma / r) ** 6
    return 4 * epsilon * (s6**2 - s6)


epsilon, sigma = 0.0104 * u.eV, 3.4 * u.angstrom


def energy(r):
    return lennard_jones(r, epsilon, sigma).sum()


forces = -units.grad(energy)(r)
print(forces)

This is the one mistake the package cannot turn into an error. If the energy is converted to a plain array first, `jax.grad` accepts it and labels the result with the wrong dimension:

In [ ]:
print(jax.grad(lambda r: energy(r).to(u.eV))(r))  # wrong: labelled Å

## JAX Transformations

The magnitudes are the only leaf of a quantity's pytree. The dimension is part of the tree structure, so `jax.jit`, `jax.vmap`, and `jax.lax.scan` accept quantities unchanged. Two quantities share a tree structure exactly when their units agree, and the traced program does not depend on the units at all:

In [ ]:
pairwise = jax.jit(jax.vmap(lambda r: lennard_jones(r, epsilon, sigma)))
print(pairwise(r))

with_units = jax.make_jaxpr(lambda a: (a * a).sum())(r)
without_units = jax.make_jaxpr(lambda a: (a * a).sum())(r.value)
print(str(with_units) == str(without_units))

## Arrays and Linear Algebra

Quantities support the usual array methods (`sum`, `mean`, `reshape`, indexing, `.at[]`). `jax.numpy` does not accept quantities, so <em>k</em>UPS provides unit-aware versions where the dimension rule is not obvious, e.g., [norm][kups.core.units.norm], [where][kups.core.units.where], [einsum][kups.core.units.einsum], and [eigh][kups.core.units.eigh]. The eigenvalues of a Hessian carry its dimension, while the eigenvectors are plain arrays.

In [ ]:
hessian = jnp.array([[2.0, 0.5], [0.5, 1.0]]) * u.eV / u.angstrom**2
eigenvalues, eigenvectors = units.eigh(hessian)
print(eigenvalues)
print(type(eigenvectors).__name__)

Any other function can be lifted with [apply][kups.core.units.apply], which keeps the dimension of its input. [relabel][kups.core.units.relabel] assigns a dimension explicitly when it cannot be inferred.

In [ ]:
print(units.apply(jnp.cumsum, r))
print(units.relabel(jnp.ones(2), units.dim_of(u.eV)))

## Static Checking

[Quantity][kups.core.units.Quantity] and [Unit][kups.core.units.Unit] are generic over the exponents of their dimension, and aliases such as `Energy` or `Velocity` name common cases. pyrefly can do arithmetic on these exponents, so it checks a function like the one below without running it. Other type checkers accept the code but do not check dimensions.

In [ ]:
def kinetic_energy(mass: units.Mass, velocity: units.Velocity) -> units.Energy:
    return 0.5 * mass * velocity**2


mass = 39.95 * u.amu  # argon
velocity = units.sqrt(kT / mass)
print(kinetic_energy(mass, velocity))

## Validated Configuration

The dimension aliases also work as [pydantic](https://docs.pydantic.dev) field types. The annotation fixes the dimension, and the input chooses the unit. Values in any compatible unit are accepted and converted, while incompatible values are rejected with the expected and received dimensions.

In [ ]:
from pydantic import BaseModel, ValidationError


class MDConfig(BaseModel):
    temperature: units.Temperature
    pressure: units.Pressure
    time_step: units.Time


config = MDConfig.model_validate(
    {"temperature": "300 K", "pressure": "1 bar", "time_step": "0.5 fs"}
)
print(config.time_step.to(u.fs))

try:
    MDConfig.model_validate(
        {"temperature": "300 eV", "pressure": "1 bar", "time_step": "0.5 fs"}
    )
except ValidationError as err:
    print(err.errors()[0]["msg"])

## Pitfalls

A quantity never converts to an array implicitly. `jnp.asarray(q)` and `q == q` raise, since both would silently drop the dimension. Use `q.to(unit)` or `q.value` to obtain magnitudes, and [testing.assert_allclose][kups.core.units.testing.assert_allclose] to compare quantities in a named unit.

In [ ]:
from kups.core.units import testing

testing.assert_allclose(1.0 * u.nm, 10.0 * u.angstrom, u.angstrom)

`jax.tree.map` treats the magnitudes as the leaf, so a nonlinear function silently keeps the old unit. Use [units.tree_map][kups.core.units.tree_map] instead, or pass `is_leaf=units.is_leaf`.

In [ ]:
area = 4.0 * u.angstrom**2
print(jax.tree.map(jnp.sqrt, area))  # wrong: labelled Å^2
print(units.tree_map(units.sqrt, area))